# plan-a-closure Tier 0 on a Colab GPU (engineering shakedown)

This runs the one-command Tier 0 pilot (`closure tier0`) on the GPU this Colab runtime has, so the pipeline is checked on
a **real device** instead of the fake NVML used in CI.

**What it is.** A shakedown on a non-target device, inside a virtual machine, with NVML as the only power source.
Precisions the GPU lacks (fp8 before compute capability 8.9) are dropped and noted in the logs.

**What it is not.** The registered trial. There is no independent power meter, so assumption A6 does not hold, and the
registered numbers come from Tier 1 on instrumented H100/H200 hardware (`docs/PREREGISTRATION.md`). Every run is
published whatever it shows (`docs/DEVIATIONS.md` has the earlier ones).

**How to run it**

1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Runtime > **Run all**.
3. Approve the Google Drive prompt in cell 1, so the results are also saved to `MyDrive/plan-a-closure-runs/`
   (decline it and they still download to this computer when the run ends).
4. Keep this tab open for about 2 hours 20 minutes. When cell 3 finishes, the results are already saved: nothing is
   lost if Colab disconnects afterwards.

Free Colab limits GPU time per day. If it says no GPU is available, try again later.

In [ ]:
# 1. Setup: the GPU, Google Drive (optional), the exact code, persistence mode.
import os, shutil, subprocess, time
REPO, REF = "https://github.com/AnonRish/plan-a-closure", "main"   # REF can be a tag or a commit, to pin the code
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU attached: Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,uuid,driver_version,memory.total,power.limit", "--format=csv"],
                     capture_output=True, text=True).stdout)
SAVE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/plan-a-closure-runs"
    os.makedirs(SAVE, exist_ok=True)
    print("results will also be saved to Google Drive:", SAVE)
except Exception as e:
    print(f"Google Drive not mounted ({type(e).__name__}): the results will download to this computer at the end")
if not os.path.isdir("/content/plan-a-closure/.git"):
    !git clone -q --depth 1 {REPO} /content/plan-a-closure
!git -C /content/plan-a-closure fetch -q --depth 1 origin {REF} && git -C /content/plan-a-closure checkout -q FETCH_HEAD
%cd /content/plan-a-closure
!pip install -q -e ".[test,hw]"
!python -c "import closure; print('plan-a-closure', closure.__version__)"
!git log -1 --format="code: commit %H (%cd)"
!nvidia-smi -pm 1 > /dev/null && echo "persistence mode on"

In [ ]:
# 2. Preflight: NVML, the energy counter, torch and CUDA, which physical GPU. FAIL lines here stop the run in cell 3;
#    WARN lines for fp8, clock sync and tcpdump are normal on Colab.
!closure preflight --gpu 0

## The run (about 2 hours 15 minutes)

Cell 3 runs Tier 0, then A Narrow Path's mock training run on the same GPU, then saves everything to Google Drive and
downloads it, all in one cell, so a dropped session after it finishes cannot lose anything. Tier 0 logs every step,
sizes the unit of work to this GPU, locks the trial's clocks at a clock the probe saw held, restores the GPU's clocks
and power limit on exit, and writes `SUMMARY.md` and `tier0_bundle.tar.gz`. The bundle leaves out the run's private
key (`signing_key.bin`) on purpose: never share that file. If the browser asks whether to allow several downloads,
allow them.

In [ ]:
# 3. The run: Tier 0, the mock training run, then save and download.
import glob, hashlib
from IPython.display import Markdown, display
OUT = time.strftime("/content/plan-a-closure/runs/tier0-%Y%m%d-%H%M%S")   # a fresh folder for every run
!closure tier0 --confirm-hardware --gpu 0 --out {OUT}
if os.path.exists(f"{OUT}/probe.json"):
    !closure treaty mock-run --device cuda:0 --probe {OUT}/probe.json --budget-flop 1e16 --dim 4096 --layers 8 --batch 2048 --out {OUT}/mock-run
keep = [p for p in (f"{OUT}/tier0_bundle.tar.gz", f"{OUT}/SUMMARY.md", f"{OUT}/mock-run/drill.json") if os.path.exists(p)]
if SAVE:
    dest = os.path.join(SAVE, os.path.basename(OUT))
    os.makedirs(dest, exist_ok=True)
    for p in keep:
        shutil.copy2(p, dest)
    print("saved to Google Drive:", dest)
if os.path.exists(f"{OUT}/tier0_bundle.tar.gz"):
    print("bundle sha256:", hashlib.sha256(open(f"{OUT}/tier0_bundle.tar.gz", "rb").read()).hexdigest())
if os.path.exists(f"{OUT}/SUMMARY.md"):
    display(Markdown(open(f"{OUT}/SUMMARY.md", encoding="utf-8").read()))
from google.colab import files
for p in keep:
    files.download(p)

## After the run

* Send back `tier0_bundle.tar.gz`, `SUMMARY.md` and `drill.json` (in your Downloads folder, and in Google Drive under
  `plan-a-closure-runs/`). On Windows, `Get-FileHash $HOME\Downloads\tier0_bundle.tar.gz` should match the sha256 printed
  above.
* Publish the result whatever it shows, labelled as a shakedown: a new section in `docs/DEVIATIONS.md` (date, GPU,
  "Google Colab VM", no independent meter, the bundle's SHA-256) and the bundle under `results/`.
* `SUMMARY.md` now reports which commit ran, the trial's SM clock, and whether the energy model drifted between
  calibration and audit (`closure drift`).
* The next step is the registered run: Tier 0 and then Tier 1 on instrumented H100/H200 hardware
  (`docs/HARDWARE_CONFIG.md`).

Optional: a two-minute rehearsal on fake hardware. Set `REHEARSE = True` and run the cell; Run all skips it.

In [ ]:
REHEARSE = False
if REHEARSE:
    !closure tier0 --mock --skip-tests --out runs/rehearsal-{int(time.time())}